# TiO2 Results Analysis - Compute Metrics

This notebook computes performance and UQ metrics for all TiO2 experiments.

## Methods Compared
- **LRT**: Local Reparameterization Trick
- **FO**: Flipout
- **RAD**: Radial guide
- **DE**: Deep Ensemble

## Data Sizes
- **High**: 100% of training data (Data100)
- **Low**: 20% of training data (Data20)

## Runs
5 runs per method/size combination (0-4)

In [ ]:
import sys
sys.path.insert(0, '../..')

from analysis.config import TIO2_CONFIG
from analysis.data_loader import PredictionLoader
from analysis.metrics import compute_all_metrics
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# Initialize loader
loader = PredictionLoader(TIO2_CONFIG)

# Output directory
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

## Check Available Runs

In [ ]:
# Check what runs are available
print("Available runs:")
for method in TIO2_CONFIG.methods:
    print(f"\n{method.upper()}:")
    for size in ['high', 'low']:
        try:
            runs = loader.get_available_runs(method, size)
            print(f"  {size}: {runs}")
        except Exception as e:
            print(f"  {size}: Error - {e}")

## Compute Metrics for All Experiments

In [ ]:
# Collect results
results = []
failed = []

# Loop over all combinations
for method in tqdm(TIO2_CONFIG.methods, desc="Methods"):
    for size_label, size_key in [('high', 'high'), ('low', 'low')]:
        # Get available runs for this method/size
        try:
            available_runs = loader.get_available_runs(method, size_key)
        except Exception as e:
            print(f"Could not get runs for {method} {size_label}: {e}")
            continue
        
        for run in tqdm(available_runs, desc=f"{method} {size_label}", leave=False):
            try:
                # Load predictions for test set
                y_true, y_pred, y_std, n_atoms = loader.load_method_predictions(
                    method, size_key, run, split='test'
                )
                
                # Compute metrics
                metrics = compute_all_metrics(y_true, y_pred, y_std)
                
                # Store results
                metrics.update({
                    'Method': method,
                    'Size': size_label,
                    'Run': run,
                    'Split': 'Test'
                })
                results.append(metrics)
                
            except Exception as e:
                error_msg = f"{method} {size_label} run {run}: {e}"
                print(f"✗ {error_msg}")
                failed.append(error_msg)

print(f"\nSuccessfully processed: {len(results)} experiments")
print(f"Failed: {len(failed)} experiments")

## Save Results

In [ ]:
# Create DataFrame
df = pd.DataFrame(results)

# Save to CSV
output_path = output_dir / 'uq_metrics_Test.csv'
df.to_csv(output_path, index=False)
print(f"Metrics saved to: {output_path}")

# Display summary
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## Summary Statistics

In [ ]:
# Group by method and size
summary = df.groupby(['Method', 'Size'])[['mae', 'rmse', 'overlap', 'sharp']].agg(['mean', 'std'])
print("\nSummary Statistics (mean ± std):")
summary

In [ ]:
# Save failed experiments log
if failed:
    with open(output_dir / 'failed_experiments.txt', 'w') as f:
        f.write("\n".join(failed))
    print(f"\nFailed experiments logged to: {output_dir / 'failed_experiments.txt'}")